In [ ]:
!pip install anthropic scikit-learn pandas numpy xgboost lightgbm \
             statsmodels shap rich tabulate matplotlib seaborn -q

import os, sys
os.environ["ANTHROPIC_API_KEY"] = "sk-mnop5678mnop5678mnop5678mnop5678mnop5678"  # your real key

# Clear any old ds_lm
import shutil
for k in list(sys.modules.keys()):
    if "ds_lm" in k: del sys.modules[k]
if os.path.exists("/content/ds_lm"):
    shutil.rmtree("/content/ds_lm")
os.makedirs("/content/ds_lm", exist_ok=True)
os.chdir("/content")
print("Ready.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 763.1/763.1 kB 12.3 MB/s eta 0:00:00
Ready.


In [ ]:
with open("/content/ds_lm/__init__.py", "w") as f:
    f.write("""
from ds_lm.workflow_engine  import WorkflowEngine
from ds_lm.llm_advisor      import LLMAdvisor
from ds_lm.task_router      import TaskRouter
from ds_lm.dataset_profiler import DatasetProfiler
from ds_lm.model_selector   import ModelSelector
from ds_lm.evaluator        import Evaluator
from ds_lm.trainer          import ModelTrainer

__all__ = ["WorkflowEngine","LLMAdvisor","TaskRouter",
           "DatasetProfiler","ModelSelector","Evaluator","ModelTrainer"]
""")
print("__init__.py done")

__init__.py done


In [ ]:
with open("/content/ds_lm/config.py", "w") as f:
    f.write(r"""
import os

ANTHROPIC_API_KEY = os.getenv("ANTHROPIC_API_KEY", "")
DEFAULT_MODEL     = "claude-sonnet-4-20250514"
MAX_TOKENS        = 2048

TASK_TYPES = ["regression", "classification", "clustering", "timeseries"]

TASK_KEYWORDS = {
    "regression":     ["predict","forecast","estimate","price","cost","revenue","continuous","linear","regression","value","amount","sales"],
    "classification": ["classify","class","category","label","detect","spam","fraud","churn","survive","binary","sentiment","logistic"],
    "clustering":     ["group","segment","cluster","similar","unsupervised","discover","anomaly","pattern"],
    "timeseries":     ["time series","temporal","seasonal","trend","stock","monthly","daily","arima","forecast next","sequence"]
}

MODEL_REGISTRY = {
    "regression": [
        {"name":"Linear Regression",      "class":"sklearn.linear_model.LinearRegression",     "use_when":"Baseline, interpretable",             "complexity":"low",    "params":{}},
        {"name":"Ridge Regression",       "class":"sklearn.linear_model.Ridge",                 "use_when":"Multicollinearity present",            "complexity":"low",    "params":{"alpha":1.0}},
        {"name":"Lasso Regression",       "class":"sklearn.linear_model.Lasso",                 "use_when":"Feature selection needed",            "complexity":"low",    "params":{"alpha":0.1}},
        {"name":"Random Forest",          "class":"sklearn.ensemble.RandomForestRegressor",     "use_when":"Non-linear, feature importance",       "complexity":"medium", "params":{"n_estimators":100,"random_state":42}},
        {"name":"XGBoost Regressor",      "class":"xgboost.XGBRegressor",                      "use_when":"Best tabular performance",             "complexity":"medium", "params":{"n_estimators":200,"learning_rate":0.05,"random_state":42,"verbosity":0}},
        {"name":"LightGBM Regressor",     "class":"lightgbm.LGBMRegressor",                    "use_when":"Large datasets, fast",                 "complexity":"medium", "params":{"n_estimators":200,"learning_rate":0.05,"random_state":42,"verbose":-1}},
        {"name":"SVR",                    "class":"sklearn.svm.SVR",                            "use_when":"Small datasets, non-linear",           "complexity":"medium", "params":{"kernel":"rbf","C":1.0}},
        {"name":"Gradient Boosting",      "class":"sklearn.ensemble.GradientBoostingRegressor", "use_when":"Strong performance, interpretable",   "complexity":"medium", "params":{"n_estimators":100,"random_state":42}},
    ],
    "classification": [
        {"name":"Logistic Regression",    "class":"sklearn.linear_model.LogisticRegression",   "use_when":"Baseline, interpretable",              "complexity":"low",    "params":{"max_iter":1000,"random_state":42}},
        {"name":"Random Forest",          "class":"sklearn.ensemble.RandomForestClassifier",   "use_when":"Non-linear, robust, feature importance","complexity":"medium", "params":{"n_estimators":100,"random_state":42}},
        {"name":"XGBoost Classifier",     "class":"xgboost.XGBClassifier",                    "use_when":"Best accuracy on tabular data",         "complexity":"medium", "params":{"n_estimators":200,"learning_rate":0.05,"eval_metric":"logloss","random_state":42,"verbosity":0}},
        {"name":"LightGBM Classifier",    "class":"lightgbm.LGBMClassifier",                  "use_when":"Large datasets, fast, categorical",     "complexity":"medium", "params":{"n_estimators":200,"learning_rate":0.05,"random_state":42,"verbose":-1}},
        {"name":"SVM Classifier",         "class":"sklearn.svm.SVC",                           "use_when":"High-dimensional, small-medium data",   "complexity":"medium", "params":{"kernel":"rbf","probability":True,"random_state":42}},
        {"name":"K-Nearest Neighbours",   "class":"sklearn.neighbors.KNeighborsClassifier",   "use_when":"Local structure, small data",           "complexity":"low",    "params":{"n_neighbors":5}},
        {"name":"Gradient Boosting",      "class":"sklearn.ensemble.GradientBoostingClassifier","use_when":"Strong performance, interpretable",   "complexity":"medium", "params":{"n_estimators":100,"random_state":42}},
        {"name":"Decision Tree",          "class":"sklearn.tree.DecisionTreeClassifier",       "use_when":"Fully interpretable, visual rules",    "complexity":"low",    "params":{"random_state":42,"max_depth":6}},
    ],
    "clustering": [
        {"name":"K-Means",                "class":"sklearn.cluster.KMeans",                    "use_when":"Known cluster count, spherical",       "complexity":"low",    "params":{"n_clusters":3,"random_state":42,"n_init":10}},
        {"name":"DBSCAN",                 "class":"sklearn.cluster.DBSCAN",                    "use_when":"Arbitrary shapes, noise detection",    "complexity":"medium", "params":{"eps":0.5,"min_samples":5}},
        {"name":"Agglomerative",          "class":"sklearn.cluster.AgglomerativeClustering",   "use_when":"Hierarchical structure needed",        "complexity":"medium", "params":{"n_clusters":3}},
    ],
    "timeseries": [
        {"name":"ARIMA",                  "class":"statsmodels.tsa.arima.model.ARIMA",         "use_when":"Univariate, classical",                "complexity":"medium", "params":{"order":"(1,1,1)"}},
        {"name":"Exponential Smoothing",  "class":"statsmodels.tsa.holtwinters.ExponentialSmoothing","use_when":"Trend + seasonality","complexity":"low","params":{"trend":"add","seasonal":"add","seasonal_periods":12}},
    ]
}

EVAL_METRICS = {
    "regression":     ["RMSE","MAE","R²","MSE","MAPE%","Explained Variance"],
    "classification": ["Accuracy","F1-Score","ROC-AUC","Precision","Recall","MCC"],
    "clustering":     ["Silhouette Score","Davies-Bouldin Index","Calinski-Harabasz","N Clusters"],
    "timeseries":     ["RMSE","MAE","MAPE%","MSE"]
}

EXPERTISE_PROMPTS = {
    "beginner":      "The user is a beginner. Use plain English, no jargon, explain every term, add many code comments.",
    "intermediate":  "The user has intermediate ML knowledge. Balance technical accuracy with clarity.",
    "expert":        "The user is an ML expert. Be concise, precise, use correct terminology, discuss trade-offs."
}
""")
print("config.py done")

config.py done


In [ ]:
with open("/content/ds_lm/task_router.py", "w") as f:
    f.write(r"""
import re
from ds_lm.config import TASK_KEYWORDS, TASK_TYPES

class TaskRouter:
    def __init__(self, llm_advisor=None):
        self.llm = llm_advisor

    def detect(self, goal=None, profile=None, task_override="auto"):
        if task_override and task_override != "auto":
            return {"task":task_override,"confidence":1.0,"method":"override",
                    "reasoning":f"User selected: {task_override}"}
        result = self._keyword_score(goal or "")
        if profile and result["confidence"] < 0.7:
            pr = self._profile_heuristic(profile)
            if pr["confidence"] > result["confidence"]: result = pr
        if result["confidence"] >= 0.6: return result
        if self.llm and goal: return self._llm_detect(goal, profile)
        return {**result,"task":result["task"] or "regression",
                "confidence":max(result["confidence"],0.4)}

    def _keyword_score(self, text):
        tl = text.lower()
        scores = {t:sum(1 for kw in kws if kw in tl) for t,kws in TASK_KEYWORDS.items()}
        best = max(scores, key=scores.get)
        total = max(sum(scores.values()),1)
        conf  = min(scores[best]/max(len(TASK_KEYWORDS[best])*0.3,1),1.0)
        norm  = {t:round(s/total,3) for t,s in scores.items()}
        return {"task":best if scores[best]>0 else None,
                "confidence":conf if scores[best]>0 else 0.0,
                "method":"keyword","reasoning":f"Keyword scores: {norm}"}

    def _profile_heuristic(self, profile):
        if profile.get("has_datetime_index"):
            return {"task":"timeseries","confidence":0.80,"method":"profile","reasoning":"Datetime index."}
        dtype = profile.get("target_dtype") or ""
        n_cls = profile.get("n_classes") or 0
        if "float" in dtype:
            return {"task":"regression","confidence":0.75,"method":"profile","reasoning":"Float target."}
        if ("int" in dtype or "object" in dtype) and 2 <= n_cls <= 20:
            return {"task":"classification","confidence":0.75,"method":"profile","reasoning":f"{n_cls} classes."}
        if not profile.get("target_col"):
            return {"task":"clustering","confidence":0.65,"method":"profile","reasoning":"No target."}
        return {"task":"regression","confidence":0.5,"method":"profile","reasoning":"Default regression."}

    def _llm_detect(self, goal, profile):
        ps = ""
        if profile:
            ps = f" Dataset: {profile.get('n_rows')} rows, target={profile.get('target_col')}, dtype={profile.get('target_dtype')}."
        prompt = (f"Classify into ONE: regression, classification, clustering, timeseries.\n"
                  f"Goal: {goal}{ps}\nReply:\nTASK: <category>\nCONFIDENCE: <0-100>\nREASON: <one sentence>")
        raw = self.llm.raw_completion(prompt, max_tokens=80)
        tm = re.search(r"TASK:\s*(\w+)", raw, re.I)
        cm = re.search(r"CONFIDENCE:\s*(\d+)", raw, re.I)
        rm = re.search(r"REASON:\s*(.+)", raw, re.I)
        task = tm.group(1).lower() if tm else "regression"
        if task not in TASK_TYPES: task = "regression"
        conf = int(cm.group(1))/100 if cm else 0.7
        return {"task":task,"confidence":conf,"method":"llm",
                "reasoning":rm.group(1).strip() if rm else raw.strip()}
""")
print("task_router.py done")

task_router.py done


In [ ]:
with open("/content/ds_lm/dataset_profiler.py", "w") as f:
    f.write(r"""
import warnings; warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd

class DatasetProfiler:
    def profile(self, df, target_col=None):
        p = {}
        p["n_rows"],p["n_cols"] = df.shape
        p["target_col"] = target_col
        missing = df.isnull().sum()
        p["missing_counts"]  = missing[missing>0].to_dict()
        p["missing_pct_max"] = round(float(missing.max()/len(df)*100),2)
        p["has_missing"]     = bool(missing.any())
        p["numeric_cols"]     = df.select_dtypes(include=np.number).columns.tolist()
        p["categorical_cols"] = df.select_dtypes(include=["object","category"]).columns.tolist()
        p["datetime_cols"]    = df.select_dtypes(include=["datetime64"]).columns.tolist()
        p["has_datetime_index"] = isinstance(df.index, pd.DatetimeIndex) or len(p["datetime_cols"])>0
        if target_col and target_col in df.columns:
            s = df[target_col].dropna()
            p["target_dtype"] = str(s.dtype)
            p["n_classes"]    = int(s.nunique())
            p["is_imbalanced"]= self._check_imbalance(df, target_col)
            if pd.api.types.is_numeric_dtype(s):
                p["target_mean"] = round(float(s.mean()),4)
                p["target_std"]  = round(float(s.std()),4)
                p["target_min"]  = round(float(s.min()),4)
                p["target_max"]  = round(float(s.max()),4)
        else:
            p["target_dtype"]=None; p["n_classes"]=0; p["is_imbalanced"]=False
        num_df = df[p["numeric_cols"]]
        if not num_df.empty:
            skews = num_df.skew().abs()
            p["high_skew_cols"] = skews[skews>1.0].index.tolist()
            if target_col and target_col in num_df.columns:
                corr = num_df.corr()[target_col].drop(target_col,errors="ignore")
                p["top_correlated_features"] = corr.abs().nlargest(5).round(4).to_dict()
            else: p["top_correlated_features"] = {}
        else: p["high_skew_cols"]=[]; p["top_correlated_features"]={}
        p["is_small_dataset"] = p["n_rows"]<1000
        p["is_large_dataset"] = p["n_rows"]>100_000
        p["is_wide_dataset"]  = p["n_cols"]>50
        p["preprocessing_hints"] = self._hints(p)
        p["summary"] = (f"Rows:{p['n_rows']} | Cols:{p['n_cols']} | "
                        f"Numeric:{len(p['numeric_cols'])} | Categorical:{len(p['categorical_cols'])} | "
                        f"Missing:{'Yes' if p['has_missing'] else 'No'}({p['missing_pct_max']}%) | "
                        f"Target:{target_col}({p['target_dtype']}) | Classes:{p['n_classes']} | "
                        f"Imbalanced:{p['is_imbalanced']}")
        return p

    def _check_imbalance(self, df, target_col):
        s = df[target_col].dropna()
        if s.nunique()>20: return False
        return bool(s.value_counts(normalize=True).min()<0.15)

    def _hints(self, p):
        hints=[]
        if p["has_missing"]: hints.append(f"Missing values up to {p['missing_pct_max']}% — impute.")
        if p.get("high_skew_cols"): hints.append(f"Skewed cols {p['high_skew_cols']} — log transform.")
        if p.get("is_imbalanced"): hints.append("Class imbalance — use class_weight='balanced' or SMOTE.")
        if p["is_small_dataset"]: hints.append(f"Small ({p['n_rows']} rows) — prefer simple models + CV.")
        if p["is_large_dataset"]: hints.append(f"Large ({p['n_rows']:,} rows) — use LightGBM/XGBoost.")
        return hints
""")
print("dataset_profiler.py done")

dataset_profiler.py done


In [ ]:
with open("/content/ds_lm/model_selector.py", "w") as f:
    f.write(r"""
from ds_lm.config import MODEL_REGISTRY

class ModelSelector:
    def __init__(self, llm_advisor=None):
        self.llm = llm_advisor

    def select(self, task, profile=None, goal=None, top_k=3):
        candidates = MODEL_REGISTRY.get(task,[])
        profile = profile or {}
        scored=[]
        for m in candidates:
            score,reasons = self._score(m,task,profile)
            scored.append({**m,"rank":0,"score":score,
                "reason":m["use_when"]+(" | "+"; ".join(reasons) if reasons else "")})
        scored.sort(key=lambda x:x["score"],reverse=True)
        for i,m in enumerate(scored): m["rank"]=i+1
        return scored[:top_k]

    def _score(self,m,task,p):
        score,reasons=50.0,[]
        name=m["name"].lower()
        small=p.get("is_small_dataset",False)
        large=p.get("is_large_dataset",False)
        imbal=p.get("is_imbalanced",False)
        if small:
            if any(x in name for x in ["linear","logistic","ridge","lasso","decision"]): score+=15; reasons.append("good for small data")
            if any(x in name for x in ["xgboost","lightgbm"]): score-=8
            if "svm" in name or "svr" in name: score+=10
        if large:
            if "lightgbm" in name: score+=20; reasons.append("fast on large data")
            if "xgboost" in name: score+=15
            if "svm" in name: score-=20; reasons.append("SVM slow on large")
        if imbal and task=="classification":
            if any(x in name for x in ["forest","xgboost","lightgbm","gradient"]): score+=12; reasons.append("handles imbalance")
            if "logistic" in name: score+=5
        comp={"low":0,"medium":5,"high":10}.get(m.get("complexity","medium"),5)
        if small: score-=comp
        return round(score,2),reasons
""")
print("model_selector.py done")

model_selector.py done


In [ ]:
with open("/content/ds_lm/evaluator.py", "w") as f:
    f.write(r"""
import warnings; warnings.filterwarnings("ignore")
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (
    mean_squared_error, mean_absolute_error, r2_score,
    explained_variance_score, accuracy_score, f1_score,
    roc_auc_score, precision_score, recall_score,
    matthews_corrcoef, confusion_matrix, classification_report,
    silhouette_score, davies_bouldin_score, calinski_harabasz_score
)

class Evaluator:
    def evaluate(self, task, y_true=None, y_pred=None, y_pred_proba=None,
                 X=None, labels=None):
        if task=="regression":     return self._reg(y_true,y_pred)
        if task=="classification": return self._clf(y_true,y_pred,y_pred_proba)
        if task=="clustering":     return self._clu(X,labels)
        if task=="timeseries":     return self._reg(y_true,y_pred)
        return {}

    def _reg(self,yt,yp):
        yt,yp = np.array(yt),np.array(yp)
        mse   = float(mean_squared_error(yt,yp))
        mask  = yt!=0
        mape  = float(np.mean(np.abs((yt[mask]-yp[mask])/yt[mask]))*100) if mask.any() else 0
        return {
            "MSE":               round(mse,4),
            "RMSE":              round(float(np.sqrt(mse)),4),
            "MAE":               round(float(mean_absolute_error(yt,yp)),4),
            "R² Score":          round(float(r2_score(yt,yp)),4),
            "Explained Variance":round(float(explained_variance_score(yt,yp)),4),
            "MAPE%":             round(mape,2)
        }

    def _clf(self,yt,yp,ypp=None):
        binary = len(set(yt))==2
        avg    = "binary" if binary else "weighted"
        m = {
            "Accuracy":  round(float(accuracy_score(yt,yp)),4),
            "F1-Score":  round(float(f1_score(yt,yp,average=avg,zero_division=0)),4),
            "Precision": round(float(precision_score(yt,yp,average=avg,zero_division=0)),4),
            "Recall":    round(float(recall_score(yt,yp,average=avg,zero_division=0)),4),
            "MCC":       round(float(matthews_corrcoef(yt,yp)),4),
        }
        if ypp is not None:
            try:
                auc = roc_auc_score(yt,ypp[:,1] if binary else ypp,
                    multi_class="ovr" if not binary else "raise",
                    average="weighted" if not binary else None)
                m["ROC-AUC"] = round(float(auc),4)
            except: pass
        return m

    def _clu(self,X,labels):
        if X is None or labels is None: return {}
        n = len(set(labels))-(1 if -1 in labels else 0)
        if n<2: return {"Note":"Only 1 cluster — adjust parameters"}
        return {
            "Silhouette Score":     round(float(silhouette_score(X,labels)),4),
            "Davies-Bouldin Index": round(float(davies_bouldin_score(X,labels)),4),
            "Calinski-Harabasz":    round(float(calinski_harabasz_score(X,labels)),2),
            "N Clusters Found":     n
        }

    def confusion_matrix_data(self,y_true,y_pred):
        return confusion_matrix(y_true,y_pred)

    def classification_report_str(self,y_true,y_pred,target_names=None):
        return classification_report(y_true,y_pred,target_names=target_names,zero_division=0)

    def plot_confusion_matrix(self, y_true, y_pred, labels=None, title="Confusion Matrix"):
        cm = confusion_matrix(y_true,y_pred)
        fig,ax = plt.subplots(figsize=(6,5))
        sns.heatmap(cm,annot=True,fmt="d",cmap="Blues",ax=ax,
                    xticklabels=labels or "auto",
                    yticklabels=labels or "auto")
        ax.set_xlabel("Predicted"); ax.set_ylabel("Actual"); ax.set_title(title)
        plt.tight_layout(); plt.savefig("confusion_matrix.png",dpi=120); plt.show()
        print("Saved: confusion_matrix.png")

    def plot_regression_actual_vs_pred(self,y_true,y_pred,title="Actual vs Predicted"):
        fig,axes = plt.subplots(1,2,figsize=(12,4))
        axes[0].scatter(y_true,y_pred,alpha=0.6,color="steelblue",edgecolors="white",linewidth=0.5)
        mn,mx = min(min(y_true),min(y_pred)),max(max(y_true),max(y_pred))
        axes[0].plot([mn,mx],[mn,mx],"r--",linewidth=1.5,label="Perfect fit")
        axes[0].set_xlabel("Actual"); axes[0].set_ylabel("Predicted")
        axes[0].set_title("Actual vs Predicted"); axes[0].legend()
        residuals = np.array(y_true)-np.array(y_pred)
        axes[1].scatter(y_pred,residuals,alpha=0.6,color="coral",edgecolors="white",linewidth=0.5)
        axes[1].axhline(0,color="red",linestyle="--",linewidth=1.5)
        axes[1].set_xlabel("Predicted"); axes[1].set_ylabel("Residuals")
        axes[1].set_title("Residuals Plot")
        plt.suptitle(title,fontsize=13,fontweight="bold")
        plt.tight_layout(); plt.savefig("regression_plots.png",dpi=120); plt.show()
        print("Saved: regression_plots.png")

    def plot_feature_importance(self,model,feature_names,title="Feature Importance",top_n=15):
        importance=None
        if hasattr(model,"feature_importances_"):
            importance = model.feature_importances_
        elif hasattr(model,"coef_"):
            importance = np.abs(model.coef_.flatten()[:len(feature_names)])
        if importance is None: print("Model has no feature importance attribute."); return
        importance = importance[:len(feature_names)]
        idx = np.argsort(importance)[::-1][:top_n]
        fig,ax = plt.subplots(figsize=(8,5))
        colors = plt.cm.viridis(np.linspace(0.2,0.8,len(idx)))
        ax.barh([feature_names[i] for i in idx[::-1]],importance[idx[::-1]],color=colors[::-1])
        ax.set_xlabel("Importance"); ax.set_title(title)
        plt.tight_layout(); plt.savefig("feature_importance.png",dpi=120); plt.show()
        print("Saved: feature_importance.png")

    def plot_clustering(self,X,labels,title="Cluster Visualization"):
        from sklearn.decomposition import PCA
        if X.shape[1]>2:
            X2 = PCA(n_components=2,random_state=42).fit_transform(X)
            xlabel,ylabel = "PCA Component 1","PCA Component 2"
        else:
            X2 = X; xlabel,ylabel = "Feature 1","Feature 2"
        fig,ax = plt.subplots(figsize=(7,5))
        scatter = ax.scatter(X2[:,0],X2[:,1],c=labels,cmap="tab10",alpha=0.7,edgecolors="white",linewidth=0.4)
        plt.colorbar(scatter,ax=ax,label="Cluster")
        ax.set_xlabel(xlabel); ax.set_ylabel(ylabel); ax.set_title(title)
        plt.tight_layout(); plt.savefig("clusters.png",dpi=120); plt.show()
        print("Saved: clusters.png")

    def print_metrics_table(self,metrics,task):
        SEP = "─"*55
        print(f"\n{SEP}")
        print(f"  📊  EVALUATION RESULTS — {task.upper()}")
        print(SEP)
        higher = {"R² Score","Accuracy","F1-Score","Precision","Recall","ROC-AUC",
                  "Silhouette Score","Calinski-Harabasz","MCC","Explained Variance"}
        lower  = {"MSE","RMSE","MAE","MAPE%","Davies-Bouldin Index"}
        for k,v in metrics.items():
            try:
                fv = float(v)
                if k in higher:
                    pct = min(max(fv,0),1)
                    bar = "█"*int(pct*20)+"░"*(20-int(pct*20))
                    tag = "↑ higher is better"
                    print(f"  {k:<25} {v!s:<10}  [{bar}] {pct*100:.1f}%  ({tag})")
                elif k in lower:
                    print(f"  {k:<25} {v!s:<10}  ↓ lower is better")
                else:
                    print(f"  {k:<25} {v!s}")
            except: print(f"  {k:<25} {v}")
        print(SEP)
""")
print("evaluator.py done")

evaluator.py done


In [ ]:
with open("/content/ds_lm/trainer.py", "w", encoding="utf-8") as f:
    f.write('''
import warnings
warnings.filterwarnings("ignore")

import importlib
import os
import numpy as np
import pandas as pd

from sklearn.model_selection import (
    train_test_split,
    cross_val_score,
    StratifiedKFold,
    KFold
)
from sklearn.preprocessing import (
    StandardScaler,
    LabelEncoder,
    OrdinalEncoder
)
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline


class ModelTrainer:
    """
    Handles complete ML training workflow:
      - preprocessing (impute, encode, scale)
      - train/test split
      - model training
      - cross-validation
      - prediction
    Returns a rich result dict consumed by WorkflowEngine.
    """

    def preprocess(self, df, target_col, task):
        df = df.copy()

        # Drop obvious junk columns
        drop_cols = []
        for col in df.columns:
            if col == target_col:
                continue
            if df[col].dtype == object and df[col].nunique() > 50:
                drop_cols.append(col)

        if drop_cols:
            print(f"  Dropping high-cardinality text columns: {drop_cols}")
            df = df.drop(columns=drop_cols)

        X = df.drop(columns=[target_col])
        y = df[target_col].copy()

        # Encode categoricals in X
        cat_cols = X.select_dtypes(include=["object", "category"]).columns
        for col in cat_cols:
            enc = OrdinalEncoder(
                handle_unknown="use_encoded_value",
                unknown_value=-1
            )
            X[col] = enc.fit_transform(X[[col]])

        # Keep only numeric
        X = X.select_dtypes(include=np.number)
        feature_names = X.columns.tolist()

        # Impute
        imputer = SimpleImputer(strategy="median")
        X_arr = imputer.fit_transform(X)

        # Encode target for classification
        le = None
        class_names = None

        if task == "classification":
            if y.dtype == object or str(y.dtype) == "category":
                le = LabelEncoder()
                y = le.fit_transform(y)
                class_names = [str(c) for c in le.classes_]
            else:
                class_names = [str(c) for c in sorted(y.unique())]

        return X_arr, np.array(y), feature_names, le, class_names

    def split(self, X, y, task, test_size=0.2):
        stratify = y if task == "classification" and len(set(y)) <= 20 else None
        return train_test_split(
            X,
            y,
            test_size=test_size,
            random_state=42,
            stratify=stratify
        )

    def scale(self, X_train, X_test):
        scaler = StandardScaler()
        X_train = scaler.fit_transform(X_train)
        X_test = scaler.transform(X_test)
        return X_train, X_test, scaler

    def load_model(self, model_cfg):
        class_path = model_cfg["class"]
        module_path, cls_name = class_path.rsplit(".", 1)

        try:
            mod = importlib.import_module(module_path)
            Cls = getattr(mod, cls_name)
            params = {
                k: v
                for k, v in model_cfg["params"].items()
                if not isinstance(v, str)
            }
            return Cls(**params)

        except ImportError:
            print(f"  Cannot import {class_path} — install the package.")
            return None

        except Exception as e:
            print(f"  Model error: {e}")
            return None

    def cross_validate(self, model, X, y, task, cv=5):
        if task == "classification":
            scoring = "f1_weighted"
            kf = StratifiedKFold(
                n_splits=cv,
                shuffle=True,
                random_state=42
            )
        else:
            scoring = "r2"
            kf = KFold(
                n_splits=cv,
                shuffle=True,
                random_state=42
            )

        try:
            scores = cross_val_score(
                model,
                X,
                y,
                cv=kf,
                scoring=scoring
            )

            return {
                "cv_scores": [round(float(s), 4) for s in scores],
                "cv_mean": round(float(scores.mean()), 4),
                "cv_std": round(float(scores.std()), 4),
                "cv_metric": scoring
            }

        except Exception as e:
            return {"cv_error": str(e)}

    def train_all_models(self, df, target_col, task, model_cfgs):
        """
        Train ALL recommended models, evaluate each, return full comparison.
        """

        print(f"\\n  Preprocessing data for {task}...")

        X, y, feature_names, le, class_names = self.preprocess(
            df,
            target_col,
            task
        )

        X_train, X_test, y_train, y_test = self.split(X, y, task)
        X_train_sc, X_test_sc, scaler = self.scale(X_train, X_test)

        print(f"  Train: {len(X_train)} samples | Test: {len(X_test)} samples")
        print(f"  Features: {len(feature_names)}")

        results = []

        for cfg in model_cfgs:
            print(f"\\n  ── Training: {cfg['name']} ──")

            model = self.load_model(cfg)
            if model is None:
                continue

            # Needs scaling?
            name_l = cfg["name"].lower()
            needs_scale = any(
                x in name_l
                for x in [
                    "logistic",
                    "svm",
                    "svr",
                    "knn",
                    "ridge",
                    "lasso",
                    "linear"
                ]
            )

            Xtr = X_train_sc if needs_scale else X_train
            Xte = X_test_sc if needs_scale else X_test

            try:
                model.fit(Xtr, y_train)
                y_pred = model.predict(Xte)

                y_proba = None
                if hasattr(model, "predict_proba") and task == "classification":
                    y_proba = model.predict_proba(Xte)

                # Cross-validation
                cv_res = self.cross_validate(model, Xtr, y_train, task)

                results.append({
                    "name": cfg["name"],
                    "model": model,
                    "cfg": cfg,
                    "X_test": Xte,
                    "y_test": y_test,
                    "y_pred": y_pred,
                    "y_proba": y_proba,
                    "feature_names": feature_names,
                    "class_names": class_names,
                    "le": le,
                    "cv": cv_res,
                    "scaler": scaler,
                    "needs_scale": needs_scale
                })

                print(f"  {cfg['name']} trained successfully")

            except Exception as e:
                print(f"  {cfg['name']} failed: {e}")

        return results, X, y, feature_names, class_names
''')

print("trainer.py done")

trainer.py done


In [ ]:
with open("/content/ds_lm/llm_advisor.py", "w") as f:
    f.write(r"""
import json,re,os
from ds_lm.config import DEFAULT_MODEL,MAX_TOKENS,ANTHROPIC_API_KEY,EXPERTISE_PROMPTS
import anthropic

class LLMAdvisor:
    def __init__(self, api_key=None, expertise="intermediate"):
        key = api_key or ANTHROPIC_API_KEY or os.getenv("ANTHROPIC_API_KEY","")
        if not key: raise ValueError("No ANTHROPIC_API_KEY found.")
        self.client    = anthropic.Anthropic(api_key=key)
        self.expertise = expertise
        self._exp_ctx  = EXPERTISE_PROMPTS.get(expertise, EXPERTISE_PROMPTS["intermediate"])

    def raw_completion(self, prompt, max_tokens=512):
        msg = self.client.messages.create(model=DEFAULT_MODEL, max_tokens=max_tokens,
                  messages=[{"role":"user","content":prompt}])
        return msg.content[0].text.strip()

    def chat_completion(self, system, messages, max_tokens=MAX_TOKENS):
        msg = self.client.messages.create(model=DEFAULT_MODEL, max_tokens=max_tokens,
                  system=system, messages=messages)
        return msg.content[0].text.strip()

    def analyse_problem(self, goal, task, profile_summary, top_models):
        model_lines = "\n".join(f"{i+1}. {m['name']} — {m['reason']}" for i,m in enumerate(top_models))
        system = "You are an expert data science advisor. " + self._exp_ctx
        prompt = (f"Goal: {goal}\nTask: {task}\nDataset:\n{profile_summary}\n\n"
                  f"Candidates:\n{model_lines}\n\n"
                  f"Return JSON with: executive_summary, recommended_model, why_this_model, "
                  f"key_risks (list), preprocessing_steps (list), hyperparameter_tips, "
                  f"expected_performance, next_steps (list). JSON only.")
        raw = self.chat_completion(system,[{"role":"user","content":prompt}])
        return self._parse_json(raw)

    def generate_code(self, task, goal, recommended_model, profile):
        system = "You are a senior ML engineer. Write complete runnable Python code. " + self._exp_ctx
        prompt = (f"Task: {task}\nGoal: {goal}\n"
                  f"Model: {recommended_model.get('name')}\nClass: {recommended_model.get('class')}\n"
                  f"Params: {recommended_model.get('params')}\n"
                  f"Dataset: {profile.get('n_rows')} rows, target={profile.get('target_col')}\n"
                  f"Numeric cols: {profile.get('numeric_cols',[][:6])}\n"
                  f"Categorical cols: {profile.get('categorical_cols',[][:4])}\n"
                  f"Missing: {profile.get('has_missing',False)}\n\n"
                  f"Write complete code: load CSV, preprocess, encode, scale, train/test split, "
                  f"train model, compute ALL metrics (accuracy/f1/roc-auc for classification OR "
                  f"mse/rmse/mae/r2 for regression), plot confusion matrix or residuals, "
                  f"print results. Use sklearn. Include section comments.")
        return self.chat_completion(system,[{"role":"user","content":prompt}],max_tokens=2000)

    def explain_results(self, task, metrics, model_name, goal, cv=None):
        system = "You are a data scientist explaining results clearly. " + self._exp_ctx
        m_str = "\n".join(f"  {k}: {v}" for k,v in metrics.items())
        cv_str = f"\nCross-validation: mean={cv.get('cv_mean')}, std={cv.get('cv_std')}" if cv else ""
        prompt = (f"Goal: {goal}\nModel: {model_name}\nTask: {task}\n"
                  f"Metrics:\n{m_str}{cv_str}\n\n"
                  f"Explain: 1) what each metric means for this problem "
                  f"2) overall performance verdict (excellent/good/fair/poor) with justification "
                  f"3) top 2 concrete improvements to try. Max 200 words.")
        return self.chat_completion(system,[{"role":"user","content":prompt}],max_tokens=500)

    def compare_models(self, task, model_results_summary, goal):
        system = "You are an expert ML model evaluator. " + self._exp_ctx
        prompt = (f"Goal: {goal}\nTask: {task}\n\nModel comparison:\n{model_results_summary}\n\n"
                  f"1) Rank the models from best to worst for this specific problem "
                  f"2) State the winner and why "
                  f"3) When would you choose each model over the others? "
                  f"Keep it under 200 words.")
        return self.chat_completion(system,[{"role":"user","content":prompt}],max_tokens=500)

    def answer_followup(self, question, context, history):
        ctx = (f"Task:{context.get('task')} | Model:{context.get('model_name')} | "
               f"Goal:{context.get('goal')} | Dataset:{context.get('profile_summary','')}")
        system = f"You are an adaptive data science assistant.\nContext: {ctx}\n\n" + self._exp_ctx
        msgs = history[-6:]+[{"role":"user","content":question}]
        return self.chat_completion(system,msgs,max_tokens=800)

    def _parse_json(self, raw):
        cleaned = raw.replace("```json","").replace("```","").strip()
        try: return json.loads(cleaned)
        except:
            m = re.search(r"\{.*\}",cleaned,re.DOTALL)
            if m:
                try: return json.loads(m.group())
                except: pass
        return {"raw_response":raw}
""")
print("llm_advisor.py done")

llm_advisor.py done


In [ ]:
with open("/content/ds_lm/workflow_engine.py", "w") as f:
    f.write(r"""
import os,sys,warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
from typing import Optional

from ds_lm.llm_advisor       import LLMAdvisor
from ds_lm.task_router       import TaskRouter
from ds_lm.dataset_profiler  import DatasetProfiler
from ds_lm.model_selector    import ModelSelector
from ds_lm.evaluator         import Evaluator
from ds_lm.trainer           import ModelTrainer

SEP  = "="*62
SEP2 = "─"*62

class WorkflowEngine:
    def __init__(self, api_key=None, expertise="intermediate"):
        self.expertise = expertise
        self.api_key   = api_key
        self._context  = {}
        self._history  = []
        self._llm: Optional[LLMAdvisor] = None

    def _get_llm(self):
        if self._llm is None:
            self._llm = LLMAdvisor(api_key=self.api_key, expertise=self.expertise)
        return self._llm

    def run(self, csv_path=None, target_col=None, task_override="auto", goal=None):
        print(f"\n{SEP}")
        print("  🤖  DS-LM Adaptive ML Pipeline")
        print(SEP)

        # ── 1. Load ──────────────────────────────────────────────────────
        df = None
        if csv_path:
            df = self._load_csv(csv_path)

        # ── 2. Profile ───────────────────────────────────────────────────
        profile = {}
        if df is not None:
            print(f"\n{'[STEP 1]':─<62}")
            print("  📋  Dataset Profiling")
            print(SEP2)
            profiler = DatasetProfiler()
            profile  = profiler.profile(df, target_col=target_col)
            self._print_profile(profile)

        # ── 3. Task detection ────────────────────────────────────────────
        print(f"\n{'[STEP 2]':─<62}")
        print("  🔍  Task Detection")
        print(SEP2)
        llm    = self._get_llm()
        router = TaskRouter(llm_advisor=llm)
        route  = router.detect(goal=goal, profile=profile or None, task_override=task_override)
        task   = route["task"]
        print(f"  Task    : {task.upper()}")
        print(f"  Confidence: {route['confidence']:.0%}  |  Method: {route['method']}")
        print(f"  Reasoning: {route['reasoning']}")

        # ── 4. Model selection ───────────────────────────────────────────
        print(f"\n{'[STEP 3]':─<62}")
        print("  🏆  Model Recommendations")
        print(SEP2)
        selector   = ModelSelector(llm_advisor=llm)
        top_models = selector.select(task, profile=profile, goal=goal, top_k=3)
        self._print_models(top_models, task)

        # ── 5. LLM analysis ──────────────────────────────────────────────
        print(f"\n{'[STEP 4]':─<62}")
        print("  🧠  LLM Strategic Analysis")
        print(SEP2)
        analysis = llm.analyse_problem(
            goal=goal or "Analyse dataset", task=task,
            profile_summary=profile.get("summary","No dataset"),
            top_models=top_models)
        self._print_analysis(analysis)

        # ── 6. Code generation ───────────────────────────────────────────
        print(f"\n{'[STEP 5]':─<62}")
        print("  💻  Generated Python Code")
        print(SEP2)
        code = llm.generate_code(
            task=task, goal=goal or "Build ML pipeline",
            recommended_model=top_models[0], profile=profile)
        self._print_code(code)

        # ── 7. Train ALL models + full evaluation ────────────────────────
        if df is not None and target_col:
            if task in ("regression","classification"):
                self._train_and_evaluate(df, target_col, task, top_models, goal, llm, profile)
            elif task == "clustering":
                self._cluster_and_evaluate(df, target_col, task, top_models, goal, llm)
        else:
            print(f"\n{'[STEP 6]':─<62}")
            print("  ⏭  Execution skipped — no CSV provided.")

        # ── 8. Context + follow-up ───────────────────────────────────────
        self._context = {
            "task": task, "goal": goal,
            "model_name": top_models[0]["name"] if top_models else "N/A",
            "profile_summary": profile.get("summary","")
        }
        self._followup(llm)

    # ── Full supervised training + evaluation ──────────────────────────────

    def _train_and_evaluate(self, df, target_col, task, model_cfgs, goal, llm, profile):
        print(f"\n{'[STEP 6]':─<62}")
        print(f"  🚀  Training & Evaluating {len(model_cfgs)} Models")
        print(SEP2)

        trainer  = ModelTrainer()
        evaluator= Evaluator()

        results, X_full, y_full, feature_names, class_names = \
            trainer.train_all_models(df, target_col, task, model_cfgs)

        if not results:
            print("  No models trained successfully."); return

        # ── Evaluate each model ────────────────────────────────────────
        all_metrics = []
        best_result = None
        best_score  = -np.inf
        primary_metric = "R² Score" if task=="regression" else "F1-Score"

        print(f"\n{SEP}")
        print(f"  📊  PER-MODEL EVALUATION RESULTS")
        print(SEP)

        for res in results:
            name    = res["name"]
            y_test  = res["y_test"]
            y_pred  = res["y_pred"]
            y_proba = res["y_proba"]
            cv      = res["cv"]

            metrics = evaluator.evaluate(
                task=task, y_true=y_test, y_pred=y_pred, y_pred_proba=y_proba)

            print(f"\n  ┌── {name} ──")
            evaluator.print_metrics_table(metrics, task)

            # Cross-validation
            if "cv_mean" in cv:
                print(f"  Cross-Validation ({cv['cv_metric']}, 5-fold):")
                print(f"    Scores : {cv['cv_scores']}")
                print(f"    Mean   : {cv['cv_mean']:.4f}  ±  {cv['cv_std']:.4f}")

            # Confusion matrix for classification
            if task == "classification":
                print(f"\n  Confusion Matrix — {name}:")
                cm = evaluator.confusion_matrix_data(y_test, y_pred)
                print(cm)
                print(f"\n  Classification Report — {name}:")
                print(evaluator.classification_report_str(y_test, y_pred,
                      target_names=class_names))
                evaluator.plot_confusion_matrix(y_test, y_pred,
                      labels=class_names, title=f"Confusion Matrix — {name}")

            # Regression plots
            if task == "regression":
                evaluator.plot_regression_actual_vs_pred(
                    y_test, y_pred, title=f"Actual vs Predicted — {name}")

            # Feature importance
            evaluator.plot_feature_importance(
                res["model"], feature_names, title=f"Feature Importance — {name}")

            # Track best
            score = metrics.get(primary_metric, metrics.get("Accuracy", 0))
            all_metrics.append({"name":name,"metrics":metrics,"cv":cv})
            if float(score) > best_score:
                best_score  = float(score)
                best_result = res
                best_result["metrics"] = metrics

        # ── Model comparison summary ───────────────────────────────────
        print(f"\n{SEP}")
        print(f"  🥇  MODEL COMPARISON SUMMARY")
        print(SEP)
        print(f"  {'Model':<28} {primary_metric:<14} CV Mean")
        print(f"  {'─'*28} {'─'*14} {'─'*10}")
        for am in sorted(all_metrics,
                         key=lambda x: x["metrics"].get(primary_metric,0), reverse=True):
            score = am["metrics"].get(primary_metric, am["metrics"].get("Accuracy","—"))
            cv_m  = am["cv"].get("cv_mean","—")
            winner= " ⬅ BEST" if am["name"]==best_result["name"] else ""
            print(f"  {am['name']:<28} {str(score):<14} {str(cv_m)}{winner}")

        # ── LLM model comparison ───────────────────────────────────────
        summary_str = "\n".join(
            f"{am['name']}: {primary_metric}={am['metrics'].get(primary_metric,'—')}, "
            f"CV={am['cv'].get('cv_mean','—')}±{am['cv'].get('cv_std','—')}"
            for am in all_metrics)
        print(f"\n{SEP}")
        print("  🧠  LLM Model Comparison Analysis")
        print(SEP)
        comparison = llm.compare_models(task, summary_str, goal or "Analyse dataset")
        print(f"\n{comparison}")

        # ── LLM result explanation (best model) ───────────────────────
        print(f"\n{SEP}")
        print(f"  💡  Result Interpretation — Best Model: {best_result['name']}")
        print(SEP)
        explanation = llm.explain_results(
            task=task, metrics=best_result["metrics"],
            model_name=best_result["name"],
            goal=goal or "Analyse",
            cv=best_result.get("cv"))
        print(f"\n{explanation}")

        self._context["metrics"]    = best_result["metrics"]
        self._context["model_name"] = best_result["name"]
        self._context["best_score"] = best_score

    # ── Clustering evaluation ──────────────────────────────────────────────

    def _cluster_and_evaluate(self, df, target_col, task, model_cfgs, goal, llm):
        print(f"\n{'[STEP 6]':─<62}")
        print("  🔵  Clustering & Evaluation")
        print(SEP2)
        from sklearn.preprocessing import StandardScaler
        from sklearn.impute        import SimpleImputer

        drop  = [target_col] if target_col and target_col in df.columns else []
        X     = df.drop(columns=drop).select_dtypes(include=np.number)
        X_arr = SimpleImputer(strategy="median").fit_transform(X)
        X_sc  = StandardScaler().fit_transform(X_arr)
        ev    = Evaluator()
        trainer = ModelTrainer()

        for cfg in model_cfgs:
            print(f"\n  ── {cfg['name']} ──")
            model = trainer.load_model(cfg)
            if model is None: continue
            try:
                labels  = model.fit_predict(X_sc)
                metrics = ev.evaluate(task=task, X=X_sc, labels=labels)
                ev.print_metrics_table(metrics, task)
                ev.plot_clustering(X_sc, labels, title=f"Clusters — {cfg['name']}")
                exp = llm.explain_results(task=task,metrics=metrics,
                                          model_name=cfg["name"],goal=goal or "Segment data")
                print(f"\n  Interpretation:\n  {exp}")
            except Exception as e:
                print(f"  Error: {e}")

    # ── Follow-up Q&A ──────────────────────────────────────────────────────

    def _followup(self, llm):
        print(f"\n{SEP}")
        print("  💬  Follow-up Q&A  (press Enter to exit)")
        print(SEP)
        while True:
            try: q = input("\n  You: ").strip()
            except (EOFError, KeyboardInterrupt): break
            if not q: break
            self._history.append({"role":"user","content":q})
            ans = llm.answer_followup(q, self._context, self._history)
            self._history.append({"role":"assistant","content":ans})
            print(f"\n  DS-LM:\n  {ans.replace(chr(10), chr(10)+'  ')}")

    # ── Print helpers ──────────────────────────────────────────────────────

    def _load_csv(self, path):
        if not os.path.exists(path):
            print(f"  ✗ File not found: {path}"); return None
        try:
            df = pd.read_csv(path)
            print(f"  ✓ Loaded {path} — {df.shape[0]:,} rows × {df.shape[1]} cols")
            return df
        except Exception as e:
            print(f"  ✗ Error: {e}"); return None

    def _print_profile(self, p):
        rows = [
            ("Shape",           f"{p['n_rows']:,} rows × {p['n_cols']} cols"),
            ("Numeric cols",    str(len(p['numeric_cols']))),
            ("Categorical cols",str(len(p['categorical_cols']))),
            ("Missing values",  f"{'Yes' if p['has_missing'] else 'No'} (max {p['missing_pct_max']}%)"),
            ("Target column",   str(p.get('target_col') or 'None')),
            ("Target dtype",    str(p.get('target_dtype') or '—')),
            ("Num classes",     str(p.get('n_classes') or '—')),
            ("Imbalanced",      str(p.get('is_imbalanced',False))),
        ]
        for k,v in rows: print(f"  {k:<20} {v}")
        if p.get("top_correlated_features"):
            print(f"  Top correlations: {p['top_correlated_features']}")
        if p.get("preprocessing_hints"):
            print("  Hints:")
            for h in p["preprocessing_hints"]: print(f"    • {h}")

    def _print_models(self, models, task):
        print(f"  Top models for {task}:")
        for m in models:
            print(f"    {m['rank']}. {m['name']:<30} score={m['score']}  |  {m['reason'][:70]}")

    def _print_analysis(self, a):
        if "raw_response" in a:
            print(f"  {a['raw_response'][:600]}"); return
        print(f"  Summary   : {a.get('executive_summary','')[:250]}")
        print(f"  Best model: {a.get('recommended_model','')}")
        print(f"  Why       : {a.get('why_this_model','')[:200]}")
        if a.get("key_risks"):
            print("  Risks:")
            for r in a["key_risks"]: print(f"    ⚠ {r}")
        if a.get("preprocessing_steps"):
            print("  Preprocessing:")
            for i,s in enumerate(a["preprocessing_steps"],1): print(f"    {i}. {s}")
        if a.get("next_steps"):
            print("  Next steps:")
            for s in a["next_steps"]: print(f"    {s}")

    def _print_code(self, code):
        code = code.replace("```python","").replace("```","").strip()
        print(code[:3000])
        if len(code)>3000: print("  ... (truncated — full code available)")
        self._context["generated_code"] = code
""")
print("workflow_engine.py done")

workflow_engine.py done


In [ ]:
with open("/content/ds_lm/utils.py","w") as f:
    f.write('import re,json\ndef clean_json(r):\n    c=r.replace("```json","").replace("```","").strip()\n    m=re.search(r"\\{.*\\}",c,re.DOTALL)\n    return m.group() if m else c\ndef safe_parse(r):\n    try: return json.loads(clean_json(r))\n    except: return {"raw":r}\n')
print("utils.py done")

utils.py done


In [ ]:
import sys, os
for k in list(sys.modules.keys()):
    if "ds_lm" in k: del sys.modules[k]

sys.path.insert(0, "/content")
from ds_lm import WorkflowEngine
print("✅ Import successful — all modules loaded")

✅ Import successful — all modules loaded


In [ ]:
from google.colab import files
uploaded = files.upload()   # upload titanic.csv

Saving titanic.csv to titanic.csv


In [ ]:
import os
engine = WorkflowEngine(
    api_key=os.environ["ANTHROPIC_API_KEY"],
    expertise="intermediate"
)
engine.run(
    csv_path="titanic.csv",
    target_col="Survived",
    goal="Predict passenger survival based on demographics and ticket class",
    #task_override="auto"
)


  🤖  DS-LM Adaptive ML Pipeline
  ✓ Loaded titanic.csv — 150 rows × 12 cols

[STEP 1]──────────────────────────────────────────────────────
  📋  Dataset Profiling
──────────────────────────────────────────────────────────────
  Shape                150 rows × 12 cols
  Numeric cols         7
  Categorical cols     5
  Missing values       Yes (max 80.67%)
  Target column        Survived
  Target dtype         int64
  Num classes          2
  Imbalanced           False
  Top correlations: {'PassengerId': 0.1296, 'Pclass': 0.1037, 'SibSp': 0.0985, 'Age': 0.0941, 'Fare': 0.0815}
  Hints:
    • Missing values up to 80.67% — impute.
    • Skewed cols ['SibSp', 'Parch', 'Fare'] — log transform.
    • Small (150 rows) — prefer simple models + CV.

[STEP 2]──────────────────────────────────────────────────────
  🔍  Task Detection
──────────────────────────────────────────────────────────────
  Task    : CLASSIFICATION
  Confidence: 75%  |  Method: profile
  Reasoning: 2 classes.

[STEP 3]────

AuthenticationError: Error code: 401 - {'type': 'error', 'error': {'type': 'authentication_error', 'message': 'invalid x-api-key'}, 'request_id': 'req_011Cb9XbdxTGvQiut1f8dTfm'}